In [1]:
import tensorflow as tf
from pathlib import Path

In [4]:
tf.random.set_seed(123)

project_dir = Path.cwd()
img_dir = project_dir / "img"

img_height = 180
img_width = 180
batch_size = 32
seed = 123

train_dataset = tf.keras.utils.image_dataset_from_directory(
    img_dir,
    validation_split=0.2,
    subset="training",
    seed=seed,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    img_dir,
    validation_split=0.2,
    subset="validation",
    seed=seed,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

class_names = train_dataset.class_names

AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.cache().prefetch(buffer_size=AUTOTUNE)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(img_height, img_width, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

model = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./255, input_shape=(img_height, img_width, 3)),
    tf.keras.layers.RandomFlip('horizontal'),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(3, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=15
)

val_loss, val_acc = model.evaluate(validation_dataset, verbose=2)

model.save('transfer_learning_model.keras')

Found 197 files belonging to 3 classes.
Using 158 files for training.
Found 197 files belonging to 3 classes.
Using 39 files for validation.


/var/folders/04/s_rk8gfn6vq1608hxwwtl_040000gn/T/ipykernel_13224/3099611878.py:36: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(


Epoch 1/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 7s 767ms/step - accuracy: 0.3734 - loss: 1.8354 - val_accuracy: 0.3077 - val_loss: 1.3391
Epoch 2/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 152ms/step - accuracy: 0.6013 - loss: 0.9762 - val_accuracy: 0.6154 - val_loss: 0.9125
Epoch 3/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 146ms/step - accuracy: 0.6266 - loss: 0.8701 - val_accuracy: 0.5897 - val_loss: 0.7643
Epoch 4/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 137ms/step - accuracy: 0.6139 - loss: 1.0320 - val_accuracy: 0.7179 - val_loss: 0.7385
Epoch 5/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 118ms/step - accuracy: 0.6835 - loss: 0.9176 - val_accuracy: 0.7436 - val_loss: 0.6648
Epoch 6/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 121ms/step - accuracy: 0.6835 - loss: 0.8181 - val_accuracy: 0.7436 - val_loss: 0.7603
Epoch 7/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 134ms/step - accuracy: 0.7532 - loss: 0.6647 - val_accuracy: 0.7949 - val_loss: 0.5052
Epoch 8/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 115ms/step - accuracy: 0.7215 - loss: 0.7442 - val_accuracy: 0.6667 - val_loss: